# Honours — Machbarkeits-Probe (Kaggle)

Prüft, was für die Titel-Ableitung verfügbar ist: Saison-Abdeckung von `games`,
die `competition_id`s der 5 Ligen / 5 Pokale / Champions League, das Format der
`round`-Werte (wie heißt das Finale?) und wie unentschiedene Finals (Elfmeter)
dargestellt sind.

**Nutzung:** Dataset `davidcariboo/player-scores` als Input hinzufügen → **Run All**
→ die komplette Ausgabe zurückmelden.

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/kaggle/input/datasets/davidcariboo/player-scores")
games = pd.read_csv(DATA / "games.csv")
comps = pd.read_csv(DATA / "competitions.csv")
apps  = pd.read_csv(DATA / "appearances.csv")

print("games season range:", games["season"].min(), "-", games["season"].max())
print("appearances season range:", apps["season"].min() if "season" in apps.columns else "(keine season-Spalte)",
      "-", apps["season"].max() if "season" in apps.columns else "")
print("appearances Spalten:", list(apps.columns))
print("games Spalten:", list(games.columns))
print("competitions Spalten:", list(comps.columns))
print("\ncompetition type-Werte:", sorted(comps[[c for c in comps.columns if 'type' in c][0]].dropna().astype(str).unique()))

import re
mask = comps["name"].str.contains("Bundesliga|Premier League|LaLiga|La Liga|Serie A|Ligue 1|DFB|FA Cup|Copa del Rey|Coppa Italia|Coupe de France|Champions", case=False, na=False)
print("\nRelevante Wettbewerbe (competition_id | type | country | name):")
tcol = [c for c in comps.columns if 'type' in c][0]
ccol = [c for c in comps.columns if 'country' in c and 'name' in c]
ccol = ccol[0] if ccol else None
for _, c in comps[mask].iterrows():
    print(" ", c["competition_id"], "|", c.get(tcol), "|", (c.get(ccol) if ccol else "?"), "|", c["name"])

print("\nround-Werte je relevantem Pokal/CL:")
cupmask = mask & comps[tcol].astype(str).str.contains("cup|pokal|champion|domestic_cup|international", case=False, na=False)
for cid in comps[cupmask]["competition_id"].head(10):
    rounds = sorted(games[games["competition_id"] == cid]["round"].dropna().astype(str).unique())
    print(" ", cid, "->", rounds[:25])

print("\nFinal-Beispiele (competition_id | season | home | away | goals | aggregate):")
fin = games[games["round"].astype(str).str.contains("final", case=False, na=False)]
cols = [c for c in ["competition_id","season","home_club_name","away_club_name","home_club_goals","away_club_goals","aggregate","aggregate_score"] if c in games.columns]
print(fin[cols].sort_values(["competition_id","season"]).tail(30).to_string(index=False))
